# H&M 2년 — M3 중심화 가치그래프 (가치축 / 가치축+반복거래축)

Dunnhumby에서 돌린 **그 설계 그대로** H&M에 적용합니다. 원형 M3(`vg_value`: `w = 1+α·log(1+지출/평균단가)`)가
아니고, 손실함수(M4)도 건드리지 않습니다. 바뀌는 것은 인접행렬 가중치뿐입니다.

`w(u,i) = exp(beta · [q_V(u)·z_V(u,i) + gamma · q_N(u)·z_N(u,i)])`, `z`는 그 고객의 지출(가치축) 또는
구매 횟수(반복거래축)에서 상품 i가 차지하는 **로그 비중**을 고객 내에서 중심화하고 다시 상품 내에서
중심화한 값이며, 가중치는 고객 쪽과 상품 쪽 여백을 번갈아 맞춰 두 쪽 평균이 모두 1이 됩니다.

**데이터 성격이 값을 정합니다(교수님 피드백).** 구조는 하나인데 H&M 자신의 통계에서 나온 값이 다릅니다 —
`beta`(가중치 변동계수 0.20에서 역산)는 가치축 **1.034**(Dunnhumby 0.390), 활동/가치 신호 표준편차 비율은
**0.404**(Dunnhumby 0.488), 고객당 엣지 중위수는 **8개**(Dunnhumby 440개)입니다.

**학습 전 게이트는 Dunnhumby와 같은 한계값을 씁니다.** 로컬에서 실제 H&M 분할로 확인한 값 —
상품가중↔가격 −0.031/−0.037, 상품가중↔인기 −0.057/−0.073, 고객가중↔degree +0.024/+0.008, 질량오차 ~1e−9.
네 값 모두 통과입니다. H&M은 주문 식별자가 없어 스키마가 **(고객, 날짜)를 구매 1건**으로 보며,
반복거래축은 그 날짜 수를 셉니다.

**비용:** epoch당 실측 240초 → arm 1개에 **약 20시간**. M1은 seed 43 학습예산 실행에서 **재사용**하므로
새 학습은 arm만입니다. arm을 나눠 돌릴 수 있고(끝난 arm은 재사용), epoch마다 저장하므로 끊겨도 이어집니다.

H&M validation(2020-09-02~08) 단일 시드이므로 **유의성·일반화·CLV 귀속을 주장하지 않고, 후보 판정도
하지 않습니다.** Dunnhumby와는 분할·평가격자가 달라 숫자를 1:1로 비교하지 않습니다. test/holdout 없음.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
from google.colab import drive

ROOT = Path('/content/drive/MyDrive/논문/data')
M1_DIR = ROOT/'results_v3_hm_clv_m2_training_budget_seed43_v1'
if not M1_DIR.is_dir():
    if os.path.ismount('/content/drive'):
        raise RuntimeError(f'Drive는 연결됐지만 M1 학습예산 결과가 없습니다: {M1_DIR}. 새 학습 안 함.')
    try:
        drive.mount('/content/drive')
    except (ValueError, NotImplementedError) as exc:
        raise RuntimeError('Drive 연결 실패. 학습은 시작되지 않았습니다.') from exc
    if not M1_DIR.is_dir():
        raise RuntimeError(f'M1 결과가 없습니다: {M1_DIR}. M1을 재학습하지 않고 중단합니다.')

SOURCE_COMMIT = '4a0dc78372d04fefc14974b90cf15481706f6b0c'
REPO = Path('/content/clv-m3-hm2y-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
if 'lightgcn_clv_v3' in sys.modules:
    assert Path(sys.modules['lightgcn_clv_v3'].__file__).resolve().parent == REPO.resolve(), \
        '다른 소스가 로드되어 있습니다. 런타임을 다시 시작하세요.'
os.chdir(REPO)
sys.path.insert(0, str(REPO))

import lightgcn_clv_m3_centered_value_graph_hm2y as hm

# arm 1개에 약 20시간이다. 주 모형은 CLV 두 축을 함께 쓰는 value_and_activity이므로 그것을 먼저 돌린다.
# 둘 다 돌리려면 ('value_and_activity','value_only'). 나눠 돌려도 끝난 arm은 재사용되므로
# 여기만 바꿔 다시 실행하면 된다. value_only는 B의 성과를 축별로 나누기 위한 분해다.
ARMS = ('value_and_activity',)
cfg = hm.configure_centered_graph_hm2y(m1_result_dir=str(M1_DIR), arms=ARMS)
print(json.dumps(hm.preflight_summary(cfg), ensure_ascii=False, indent=2))

## 1. M1 곡선 재사용 확인 (학습 없음)
seed 43 학습예산 실행의 M1 곡선을 쓰되, seed·차원·L2·batch·lr·음성 수·epoch·평가격자가 모두 일치할 때만
받아들입니다. 하나라도 다르면 여기서 멈춥니다 — M1을 몰래 다시 학습하지 않습니다.

In [ ]:
baseline = hm.load_baseline_curve(cfg)
print('M1 평가 epoch:', [r['epoch'] for r in baseline], '| 지표 수:', len(baseline[0]['metrics']))
for r in baseline:
    m = r['metrics']
    print(f"  ep{r['epoch']:3d} Recall@10 {m['recall@10']:.6f} | "
          f"가중 적중값@10 {m['price_purchase_amount_weighted_hit@10']:.6f}")

## 2. 엣지 가중치 게이트 (학습 없음, 약 10분)
`beta`를 역산하고 가중치를 감사합니다. 가격·인기·degree 상관이 한계를 넘으면 예외가 나고 학습은
시작되지 않습니다. 로컬 확인값과 크게 다르면 입력이 달라진 것이므로 멈추고 확인하세요.

| arm | beta | CV | 상품↔가격 | 상품↔인기 | 고객↔degree |
|---|---|---|---|---|---|
| 가치+반복거래 (주 모형) | 0.609 | 0.200 | −0.037 | −0.073 | +0.008 |
| 가치축만 (분해용) | 1.034 | 0.200 | −0.031 | −0.057 | +0.024 |

In [ ]:
import pandas as pd
prepared = hm._prepare(cfg)
graphs = {}
for spec in hm.arm_specifications(cfg):
    graphs[spec['model_id']] = hm.build_arm_graph(prepared, cfg, spec)   # 게이트 실패 시 예외
print(pd.DataFrame([{'arm': spec['arm'], 'beta': graphs[spec['model_id']]['beta'],
                     **graphs[spec['model_id']]['audit']}
                    for spec in hm.arm_specifications(cfg)]).to_string(index=False))

## 3. 학습 (arm 1개당 약 20시간)
epoch마다 Drive에 저장하므로 끊긴 뒤 같은 설정으로 다시 실행하면 이어집니다. 완료된 arm은 건너뜁니다.
평가는 100·200·300 epoch에서만 합니다(고객 67,000명 전체 채점이 비싸서 H&M은 원래 이 격자를 씁니다).
`ARMS`를 바꿔 나머지 arm을 나중에 돌려도 이미 끝난 arm은 다시 학습하지 않습니다.

In [ ]:
import torch
assert torch.cuda.is_available(), '전체 실험은 GPU 런타임을 사용하세요.'
curve = hm.run_centered_graph_hm2y(cfg)

## 4. 판독표와 원본 ZIP
`difference`는 보고 epoch(100·300)에서 arm−M1이고, 두 arm이 모두 끝났으면 B−A도 포함됩니다.
**단일 시드이므로 방향만 기록하고 후보 판정은 하지 않습니다**(`candidate_decided`는 항상 false).
미리보기만으로 판단하지 말고 ZIP의 전체·세그먼트 지표를 함께 보세요.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
difference = hm.difference_frame(curve)
paths = curve.attrs['result_paths']
metrics = ['recall@10', 'ndcg@10', 'recall@20', 'ndcg@20', 'recall@50', 'ndcg@50',
           'price_purchase_amount_weighted_hit@10', 'vndcg@10',
           'price_purchase_amount_weighted_hit@50', 'vndcg@50']
print(difference.set_index(['model_id', 'reference', 'epoch'])[metrics].to_string())
print()
print(json.dumps(curve.attrs['reading'], ensure_ascii=False, indent=2))

zip_path = Path('/content/m3_centered_value_graph_hm2y_results.zip')
with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as archive:
    for path in paths.values():
        archive.write(path, arcname=Path(path).name)
files.download(str(zip_path))